# The Gender Earnings Gap — Which Two Numbers You Divide

The commonly quoted figure is about eighty-two cents on the dollar. This notebook computes it
from ACS table `B20017` and gets **80.5 cents** — and then computes it twice more, from the
same table, for the same states, in the same year, and gets **72.4 cents** and **87.7 cents**.

Nothing is wrong with any of the three. They are the ratio of female to male median earnings
among full-time year-round workers, among all people with earnings, and among everyone else.
All three are published. All three are the gender pay gap to somebody. **The choice of
denominator is worth 8.1 cents**, which is larger than the spread of the headline figure
across most of the country, and it moves in the same direction in **51 of 51 states**.

The notebook then asks the question that choice invites. The pooled ratio, 72.4 cents, is
*below both* of its components. For means that would prove the two groups are mixed differently
by sex; for medians it proves nothing, so `B20017` cannot settle it. `B20005` can, because it
publishes the counts `B20017` omits: men are in the full-time year-round group at **71.1%**
against women's **60.1%**, a gap that holds in every state, and women are **1.38×** as likely
as men to be outside it. Conditioning on full-time year-round work is conditioning on something
that itself differs by sex by eleven percentage points.

Finally it asks whether the states can be ranked. They cannot — **0 of 51** — and this case
answers a question handbook §7.3 leaves explicitly open, because its estimand is the first in
that series whose between-unit spread is small relative to its own scale. It also shows *why*
the answer is zero in a way the earlier cases could not: **79.6% of the 1,275 state pairs are
statistically separated**, and a rank still needs all fifty of its comparisons to succeed.

## Learning Objectives

1. **Locate a headline statistic's definitional choice**, compute the alternatives the same
   source publishes, and report what the choice is worth.
2. **Use the ratio MOE, not the proportion MOE**, and know which one a pair of estimates calls
   for.
3. **Recognise when arithmetic is suggestive rather than probative** — a pooled ratio below both
   components proves composition for means and proves nothing for medians — and go and find the
   table that settles it.
4. **See that a conditioning variable can be part of what is being measured.**
5. **Extend the precision-and-rank result** to an estimand whose spread is small relative to its
   level, and read the pairwise separation rate against the number of determined ranks.

## Prerequisites

- Handbook §1.1 (ACS margins of error) — the 90% convention, and why `SE = MOE / 1.645`.
- Handbook §7.3 (the precision-and-rank series) — this notebook adds a row to that table and
  answers one of the open questions it names.
- Handbook §3 (denominators, universes and partitions) — the corpus's most frequently confirmed
  failure mode, and the reason sections 2 through 4 exist.

## Data Sources

**U.S. Census Bureau, American Community Survey 5-year estimates, 2023**, retrieved live.

- **`B20017`** — *Median earnings in the past 12 months by sex by work experience.* Six
  estimates: male and female medians for all earners, for full-time year-round workers, and for
  everyone else, each with a published 90% margin of error. It publishes **no counts**.
- **`B20005`** — *Sex by work experience by earnings.* The counts `B20017` omits, used in
  section 5 and nowhere else.

Both are pinned to a dataset year. ACS 5-year releases are not revised, so the pin is stable
(handbook §5.4). Variable labels are carried verbatim into the provenance record, so the
semantics of every column can be checked without leaving the notebook.

Puerto Rico appears in the `state:*` universe and is not a state. It is dropped by name, and
the drop is printed rather than assumed.

## 1. Setup

Note which margin-of-error formula is used and why. Census publishes two, and they are not
interchangeable:

- the **proportion** formula, when the numerator is a subset of the denominator;
- the **ratio** formula, when it is not.

Female and male earners are disjoint groups. Every interval below therefore uses the ratio
formula, `MOE(r) = sqrt(MOE(X)² + r²·MOE(Y)²) / Y`. Using the proportion formula here would
subtract a term that does not belong and report intervals that are too narrow.

In [1]:
"""Configuration."""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.reliability import rank_intervals
from krl_tutorials_data.sources import fetch_acs_wide
from krl_tutorials_data.uncertainty import Estimate, Z90, ratio_moe

#: Pinned vintage. ACS 5-year releases are not revised, so this is a stable pin.
YEAR = 2023

#: B20017 -- median earnings by sex and work status. Labels are verbatim from
#: the Census variable metadata and are carried into provenance, so the
#: semantics of every column can be checked without leaving this notebook.
EARNINGS = {
    "B20017_002E": "Median earnings -- Male -- Total",
    "B20017_002M": "Median earnings -- Male -- Total, MOE",
    "B20017_003E": "Median earnings -- Male -- Worked full-time, year-round",
    "B20017_003M": "Median earnings -- Male -- Worked full-time, year-round, MOE",
    "B20017_004E": "Median earnings -- Male -- Other",
    "B20017_004M": "Median earnings -- Male -- Other, MOE",
    "B20017_005E": "Median earnings -- Female -- Total",
    "B20017_005M": "Median earnings -- Female -- Total, MOE",
    "B20017_006E": "Median earnings -- Female -- Worked full-time, year-round",
    "B20017_006M": "Median earnings -- Female -- Worked full-time, year-round, MOE",
    "B20017_007E": "Median earnings -- Female -- Other",
    "B20017_007M": "Median earnings -- Female -- Other, MOE",
}

#: B20005 -- counts of earners by sex and work status. B20017 publishes medians
#: and no counts, so composition cannot be established from it alone.
COUNTS = {
    "B20005_005E": "Male -- full-time, year-round -- with earnings",
    "B20005_005M": "Male -- full-time, year-round -- with earnings, MOE",
    "B20005_028E": "Male -- other -- with earnings",
    "B20005_028M": "Male -- other -- with earnings, MOE",
    "B20005_052E": "Female -- full-time, year-round -- with earnings",
    "B20005_052M": "Female -- full-time, year-round -- with earnings, MOE",
    "B20005_075E": "Female -- other -- with earnings",
    "B20005_075M": "Female -- other -- with earnings, MOE",
}

#: (female variable, male variable) for each published way of forming the ratio.
#: Every one of these is "the gender pay gap" to somebody.
DEFINITIONS = {
    "full-time, year-round": ("B20017_006E", "B20017_006M", "B20017_003E", "B20017_003M"),
    "all with earnings": ("B20017_005E", "B20017_005M", "B20017_002E", "B20017_002M"),
    "not full-time, year-round": ("B20017_007E", "B20017_007M", "B20017_004E", "B20017_004M"),
}

#: The definition the headline "82 cents" figure uses, and this notebook's spine.
HEADLINE = "full-time, year-round"

Z95 = 1.96                 # the interval convention section 5 onward reports
RELIABILITY_FLOOR = 0.7    # the publishing convention, handbook 7.1

pd.set_option("display.width", 112)
print(f"vintage     : ACS 5-year {YEAR}")
print(f"definitions : {', '.join(DEFINITIONS)}")
print(f"headline    : {HEADLINE}")


vintage     : ACS 5-year 2023
definitions : full-time, year-round, all with earnings, not full-time, year-round
headline    : full-time, year-round


In [2]:
"""Retrieve medians and counts."""
earnings, earnings_prov = fetch_acs_wide(
    year=YEAR, variables=EARNINGS, geography="state:*"
)
counts, counts_prov = fetch_acs_wide(
    year=YEAR, variables=COUNTS, geography="state:*"
)

# Puerto Rico is in the state:* universe and is not a state. It is dropped by
# name rather than by row position, and the drop is reported rather than silent.
def _states_only(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[frame["NAME"] != "Puerto Rico"].reset_index(drop=True)

dropped = sorted(set(earnings["NAME"]) - set(_states_only(earnings)["NAME"]))
earnings, counts = _states_only(earnings), _states_only(counts)
assert (earnings["NAME"].values == counts["NAME"].values).all(), (
    "the two tables must align row-for-row before any ratio is formed"
)

print(f"B20017 : {earnings.shape[0]} units, {len([c for c in EARNINGS if c.endswith('E')])} estimates")
print(f"B20005 : {counts.shape[0]} units")
print(f"dropped from the state:* universe: {', '.join(dropped)}")
print(f"payload sha256 (B20017): {earnings_prov.content_sha256[:16]}...")
print(f"payload sha256 (B20005): {counts_prov.content_sha256[:16]}...")


B20017 : 51 units, 6 estimates
B20005 : 51 units
dropped from the state:* universe: Puerto Rico
payload sha256 (B20017): 0011e2518da1a771...
payload sha256 (B20005): 1830cfae8e09c398...


## 2. One question, three published answers

Each row is a ratio of female to male median earnings. Same table, same states, same year.

In [3]:
"""One question, three published answers."""
def ratio_frame(female_e, female_m, male_e, male_m):
    """Point ratio and its Census ratio MOE, per state.

    The **ratio** formula, not the proportion formula: female and male earners
    are disjoint groups, so the numerator is not a subset of the denominator.
    """
    value = (earnings[female_e] / earnings[male_e]).to_numpy(float)
    moe = np.array([
        ratio_moe(
            Estimate(value=earnings.loc[i, female_e], moe=earnings.loc[i, female_m]),
            Estimate(value=earnings.loc[i, male_e], moe=earnings.loc[i, male_m]),
        )
        for i in earnings.index
    ])
    return value, moe

ratios = {}
rows = []
for label, (fe, fm, me, mm) in DEFINITIONS.items():
    value, moe = ratio_frame(fe, fm, me, mm)
    ratios[label] = (value, moe)
    rows.append({
        "definition": label,
        "median state": np.median(value),
        "in cents": f"{100 * np.median(value):.1f}c",
        "lowest": f"{earnings.loc[int(np.argmin(value)), 'NAME']} {np.min(value):.3f}",
        "highest": f"{earnings.loc[int(np.argmax(value)), 'NAME']} {np.max(value):.3f}",
        "median MOE": f"+/-{np.median(moe):.4f}",
    })
print(pd.DataFrame(rows).to_string(index=False, formatters={"median state": "{:.4f}".format}))


               definition median state in cents        lowest                    highest median MOE
    full-time, year-round       0.8046    80.5c Wyoming 0.704              Vermont 0.896  +/-0.0087
        all with earnings       0.7237    72.4c    Utah 0.590 District of Columbia 0.855  +/-0.0082
not full-time, year-round       0.8773    87.7c   Texas 0.771             Delaware 1.004  +/-0.0253


**80.5, 72.4 and 87.7 cents.** The first is the figure almost every headline
means. The second is what happens if you stop conditioning on full-time year-round work. The
third is the ratio among everyone the first one excludes — and it is *higher* than the headline,
which is the opposite of what most readers would guess.

The state ranges are also worth noting. Under the headline definition the lowest state is
Wyoming at 0.704 and the highest Vermont at 0.896. Under the third, Delaware comes out at
**1.004**: among workers who were not full-time year-round, the female median exceeds the male
median. That is not an error, and it is not a claim about equality; it is a statement about two
medians in a population that is defined by not working full time.

## 3. What the denominator is worth

The two definitions a reader is most likely to encounter differ by more than most of the
between-state variation people argue about.

In [4]:
"""How much of the headline is the choice of denominator?"""
ftyr = ratios["full-time, year-round"][0]
allw = ratios["all with earnings"][0]
other = ratios["not full-time, year-round"][0]
shift = ftyr - allw

print(f"full-time, year-round      : {100 * np.median(ftyr):.1f} cents")
print(f"all with earnings          : {100 * np.median(allw):.1f} cents")
print(f"difference                 : {100 * np.median(shift):.1f} cents, "
      f"from the denominator alone")
print()
print(f"the full-time definition is the more favourable one in "
      f"{int((shift > 0).sum())} of {len(shift)} states")
print(f"  smallest shift : {earnings.loc[int(np.argmin(shift)), 'NAME']} "
      f"{100 * np.min(shift):+.1f} cents")
print(f"  largest shift  : {earnings.loc[int(np.argmax(shift)), 'NAME']} "
      f"{100 * np.max(shift):+.1f} cents")
print()
print("and the third definition points the other way:")
print(f"  not full-time, year-round : {100 * np.median(other):.1f} cents -- "
      f"{100 * (np.median(other) - np.median(ftyr)):+.1f} cents against the headline")


full-time, year-round      : 80.5 cents
all with earnings          : 72.4 cents
difference                 : 8.1 cents, from the denominator alone

the full-time definition is the more favourable one in 51 of 51 states
  smallest shift : District of Columbia +1.3 cents
  largest shift  : Idaho +12.7 cents

and the third definition points the other way:
  not full-time, year-round : 87.7 cents -- +7.3 cents against the headline


**8.1 cents**, and it is not noise: the full-time definition is the more
favourable of the two in **51 of 51 states**. A systematic shift in one direction in every
single unit is not a sampling artifact.

The size of the shift varies a great deal — **+1.3 cents** in the District of Columbia,
**+12.7 cents** in Idaho — which means the choice of definition also reorders states. A
comparison between two places is not safe unless both are computed the same way, and the
published tables make it easy to mix them.

**[rule]** A headline statistic that a source publishes more than one way is a choice before
it is a measurement. State which one you took and what the others give.

## 4. An arithmetic fact that needs explaining

Put the three ratios side by side and something is odd.

In [5]:
"""An arithmetic fact that needs explaining."""
print(f"full-time, year-round      : {np.median(ftyr):.4f}")
print(f"not full-time, year-round  : {np.median(other):.4f}")
print(f"all with earnings          : {np.median(allw):.4f}")
print()
print(f"The pooled ratio is BELOW both of its components, by "
      f"{np.median(ftyr) - np.median(allw):.4f} and "
      f"{np.median(other) - np.median(allw):.4f} respectively.")
print()
print("For means this would require the two groups to be mixed in different")
print("proportions by sex. These are medians, for which no such decomposition")
print("holds, so the arithmetic is suggestive and not a proof. B20005 publishes")
print("the counts B20017 does not, and settles it.")


full-time, year-round      : 0.8046
not full-time, year-round  : 0.8773
all with earnings          : 0.7237

The pooled ratio is BELOW both of its components, by 0.0809 and 0.1535 respectively.

For means this would require the two groups to be mixed in different
proportions by sex. These are medians, for which no such decomposition
holds, so the arithmetic is suggestive and not a proof. B20005 publishes
the counts B20017 does not, and settles it.


The pooled ratio is **below both** of its components — 0.0809 below the
full-time figure and 0.1535 below the other one.

For **means**, this is a familiar structure and it would be self-proving: a weighted average
cannot fall outside the range of the things being averaged unless the weights differ between the
two populations being compared. Seeing it would establish that women and men are distributed
differently between the two groups.

These are **medians**. A median of a mixture is not a weighted average of the subgroup medians,
and no decomposition of this kind holds. So the arithmetic is **suggestive and not probative**,
and `B20017` cannot close the question, because it publishes no counts at all.

**[rule]** When an observation about a statistic depends on a decomposition, check that the
decomposition applies to that statistic. Medians do not obey the identities means obey, and the
intuition transfers long before the algebra does.

## 5. The counts, from the table that has them

`B20005` publishes what `B20017` omits: how many men and women are in each work-status group.

In [6]:
"""Who is in the full-time, year-round group?"""
male_ft, male_other = counts["B20005_005E"], counts["B20005_028E"]
female_ft, female_other = counts["B20005_052E"], counts["B20005_075E"]
male_share = (male_ft / (male_ft + male_other)).to_numpy(float)
female_share = (female_ft / (female_ft + female_other)).to_numpy(float)
share_gap = male_share - female_share

summary = pd.DataFrame({
    "median": [np.median(male_share), np.median(female_share), np.median(share_gap)],
    "minimum": [male_share.min(), female_share.min(), share_gap.min()],
    "maximum": [male_share.max(), female_share.max(), share_gap.max()],
}, index=["men, FTYR share", "women, FTYR share", "gap (men - women)"])
print(summary.to_string(float_format=lambda v: f"{v:.4f}"))
print()
print(f"men's full-time year-round share exceeds women's in "
      f"{int((share_gap > 0).sum())} of {len(share_gap)} states")
print(f"  narrowest : {counts.loc[int(np.argmin(share_gap)), 'NAME']} {np.min(share_gap):+.4f}")
print(f"  widest    : {counts.loc[int(np.argmax(share_gap)), 'NAME']} {np.max(share_gap):+.4f}")

pooled_male = male_ft.sum() / (male_ft.sum() + male_other.sum())
pooled_female = female_ft.sum() / (female_ft.sum() + female_other.sum())
print()
print(f"pooled over states: men {pooled_male:.4f}, women {pooled_female:.4f}")
print(f"women are {(1 - pooled_female) / (1 - pooled_male):.2f}x as likely as men "
      f"to be outside the full-time, year-round group")


                   median  minimum  maximum
men, FTYR share    0.7091   0.6400   0.7428
women, FTYR share  0.5980   0.5128   0.7019
gap (men - women)  0.1130   0.0320   0.1910

men's full-time year-round share exceeds women's in 51 of 51 states
  narrowest : District of Columbia +0.0320
  widest    : Utah +0.1910

pooled over states: men 0.7106, women 0.6008
women are 1.38x as likely as men to be outside the full-time, year-round group


**Men are in the full-time year-round group at a median 70.9% against women's
59.8%** — a gap of **11.3 percentage points**, and it is positive in **51 of 51 states**. Pooled
across states the figures are **71.1%** and **60.1%**, which makes women **1.38×** as likely as
men to sit outside the group the headline statistic is computed on.

That settles section 4: the two groups genuinely are mixed differently by sex, and the pooled
ratio falls below both components because the lower-earning group is disproportionately female.

It also says something the ratio itself cannot. The headline number answers the question *among
people who worked full-time year-round, what is the ratio of female to male median earnings* —
and it reaches that question by conditioning on a variable that differs by sex by eleven points.
Whether that conditioning is right depends entirely on what is being asked. For *equal pay for
equal work* it removes a source of variation you may want removed. For *what women earn relative
to men* it removes part of the phenomenon.

**[rule]** Before conditioning, check whether the conditioning variable differs across the groups
being compared. If it does, the conditioned statistic answers a narrower question than its name
suggests, and the name will travel further than the caveat.

The spread is large and worth reporting: the sex gap in full-time year-round share runs from
**+3.2 points** in the District of Columbia to **+19.1 points** in Utah. Whatever the headline
ratio measures, it is not measuring the same thing in those two places.

## 6. Reliability, and a case handbook §7.3 asks for

§7.3 tracks how much precision is needed before a ranking is determined, across three notebooks.
It closes by naming what is not established: *"no case in the series has an estimand whose
between-unit spread is small relative to its own scale — where the result should, if anything, be
worse."*

This estimand is exactly that. State ratios run from 0.70 to 0.90 — a spread of about a fifth of
the level itself. §7.3 is the authority on how that compares with the earlier three; this
notebook supplies the number and leaves the comparison there. The construction below is D21's,
unchanged, so the rows compare.

In [7]:
"""Reliability and rank identifiability on the headline ratio."""
value, moe = ratios[HEADLINE]
error = moe / Z90          # the published MOE is a 90% one
n_units = len(value)

# DerSimonian-Laird moment estimator for the between-unit variance -- the same
# construction handbook 7.3 uses for D04, D10 and D21, so the rows compare.
weight = 1.0 / error ** 2
pooled = (weight * value).sum() / weight.sum()
q_statistic = (weight * (value - pooled) ** 2).sum()
scale = weight.sum() - (weight ** 2).sum() / weight.sum()
tau_squared = max(0.0, (q_statistic - (n_units - 1)) / scale)
reliability = tau_squared / (tau_squared + error ** 2)

spread = value.max() - value.min()
half_width = Z95 * np.median(error)

print(f"units                                        : {n_units}")
print(f"between-state spread                         : {spread:.4f} "
      f"({value.min():.4f} to {value.max():.4f})")
print(f"spread as a fraction of the mean level       : {spread / value.mean():.4f}")
print(f"median standard error                        : {np.median(error):.5f}")
print(f"median 95% interval width                    : {2 * half_width:.4f}")
print(f"interval width as a fraction of the spread   : {2 * half_width / spread:.4f}")
print()
print(f"between-unit standard deviation tau          : {np.sqrt(tau_squared):.5f}")
print(f"reliability lambda, median                   : {np.median(reliability):.4f}")
print(f"reliability lambda, minimum                  : {reliability.min():.4f}")
print(f"share clearing the {RELIABILITY_FLOOR} publishing convention: "
      f"{(reliability >= RELIABILITY_FLOOR).mean():.1%}")


units                                        : 51
between-state spread                         : 0.1928 (0.7036 to 0.8964)
spread as a fraction of the mean level       : 0.2383
median standard error                        : 0.00530
median 95% interval width                    : 0.0208
interval width as a fraction of the spread   : 0.1078

between-unit standard deviation tau          : 0.03276
reliability lambda, median                   : 0.9745
reliability lambda, minimum                  : 0.8310
share clearing the 0.7 publishing convention: 100.0%


A **spread of 0.1928 on a mean level of about 0.81 — a ratio of 0.2383**, the
narrowest relative spread in the series. The measurement is nonetheless precise: the median 95%
interval is **0.0208 wide, 10.78%** of the between-state spread, and reliability λ has a median
of **0.9745**, a minimum of **0.8310**, and clears the 0.7 publishing convention in **100.0%** of
states.

By every precision criterion this is a good case. It has the fewest units in the series — **51**,
against 133 and 3,018 — so it also faces the least multiplicity pressure of any case tested.

## 7. The ranking

In [8]:
"""Can the states be put in order?"""
estimates = {
    earnings.loc[i, "NAME"]: Estimate(value=value[i], moe=Z95 * error[i])
    for i in range(n_units)
}
intervals = rank_intervals(estimates)
widths = np.array([r.width for r in intervals])
determined = [r for r in intervals if r.is_determined]

print(f"ranks determined out of {n_units:<27}: {len(determined)}")
print(f"median attainable span                       : {int(np.median(widths))} "
      f"({np.median(widths) / n_units:.1%} of the table)")
print(f"narrowest span                               : {widths.min()}")
print(f"spans under 10                               : {int((widths < 10).sum())}")
print()
tightest = sorted(intervals, key=lambda r: (r.width, r.point))[:6]
print(pd.DataFrame([
    {"state": r.label, "rank by point": r.point, "best": r.best,
     "worst": r.worst, "span": r.width}
    for r in tightest
]).to_string(index=False))


ranks determined out of 51                         : 0
median attainable span                       : 19 (37.3% of the table)
narrowest span                               : 3
spans under 10                               : 9

    state  rank by point  best  worst  span
     Utah             50    49     51     3
  Wyoming             51    49     51     3
Louisiana             49    48     51     4
  Vermont              1     1      5     5
 New York              2     1      6     6
  Alabama             48    44     49     6


**Zero of fifty-one.** Not one state's position in the table is fixed by the
published intervals, and the median state could occupy any of **19** ranks — **37.3%** of the
table.

This is the fourth case in §7.3's series and the most favourable one on every axis except the
estimand's relative spread: the highest reliability except D21, an interval
**eight times** tighter than D10's as a fraction of spread, and a sixtieth of D21's unit
count. It produces the same answer the other
three did.

**More units makes ranking worse; fewer units does not rescue it, and neither does precision.**

The extremes behave as D21's rule predicts. **9** states have an attainable span under 10, and
the tightest are **Utah and Wyoming at span 3** — both at the bottom of the table — with Vermont
at the top able to occupy ranks 1 through 5. *You may name the extremes; you may not publish the
list.*

## 8. Why zero, when four pairs in five are separated?

The earlier cases in the series reported determined ranks without reporting the pairwise
separation rate, which leaves the zero looking like a precision failure. It is not.

In [9]:
"""Why does high pairwise separation still give no determined ranks?"""
pairs = list(itertools.combinations(range(n_units), 2))
z_scores = np.array([
    abs(value[i] - value[j]) / np.sqrt(error[i] ** 2 + error[j] ** 2) for i, j in pairs
])
separated_90 = int((z_scores > Z90).sum())
separated_95 = int((z_scores > Z95).sum())

print(f"pairs                                        : {len(pairs):,}")
print(f"separated at 90%                             : {separated_90:,} "
      f"({separated_90 / len(pairs):.1%})")
print(f"separated at 95%                             : {separated_95:,} "
      f"({separated_95 / len(pairs):.1%})")
print()
rate = separated_90 / len(pairs)
comparisons = n_units - 1
expected = n_units * rate ** comparisons
print(f"a determined rank requires all {comparisons} of a state's comparisons to succeed.")
print(f"treating them as independent -- which they are not -- {rate:.3f}^{comparisons} "
      f"= {rate ** comparisons:.2e},")
print(f"so the expected number of determined ranks is about {expected:.4f}.")
print(f"observed: {len(determined)}.")
print()
print("The heuristic is not a proof: the comparisons share estimates and are")
print("positively dependent, which makes determined ranks more likely than this")
print("figure suggests, not less. It is offered as an order of magnitude for why")
print("a high pairwise separation rate and zero determined ranks coexist.")


pairs                                        : 1,275
separated at 90%                             : 1,015 (79.6%)
separated at 95%                             : 970 (76.1%)

a determined rank requires all 50 of a state's comparisons to succeed.
treating them as independent -- which they are not -- 0.796^50 = 1.12e-05,
so the expected number of determined ranks is about 0.0006.
observed: 0.

The heuristic is not a proof: the comparisons share estimates and are
positively dependent, which makes determined ranks more likely than this
figure suggests, not less. It is offered as an order of magnitude for why
a high pairwise separation rate and zero determined ranks coexist.


**1,015 of 1,275 state pairs — 79.6% — are statistically separated at 90%**, and
**76.1%** at 95%. On any pairwise question this measurement performs well: pick two states at
random and four times in five you can say which is higher.

A determined rank is a different object. It requires a state to be separated from **all fifty**
of the others, not from one. At a 79.6% per-comparison rate, and treating comparisons as
independent, that is 0.796⁵⁰ ≈ **1.12 × 10⁻⁵** — about **0.0006** determined ranks expected
across the table. Observed: **0**.

The independence assumption is wrong, and wrong in the direction that matters: the fifty
comparisons share one estimate and are positively dependent, which makes determined ranks *more*
likely than the figure suggests, not less. It is an order of magnitude, not a proof. But it is
enough to show that the zero is arithmetic rather than imprecision — and the same arithmetic
explains why adding units makes things worse, since the exponent is the unit count.

**[rule]** Report the pairwise separation rate beside the count of determined ranks. One number
without the other reads as a failed measurement instead of a structural result.

## 9. What survives

Ranking fails. Two weaker questions do not.

In [10]:
"""What survives: detection against a benchmark, and the ends of the table."""
benchmark = pooled          # the precision-weighted national ratio
detach_95 = np.abs(value - benchmark) / error > Z95
detach_998 = np.abs(value - benchmark) / error > 3.09

print(f"precision-weighted national ratio            : {benchmark:.4f}")
print(f"states separating from it at 95%             : {int(detach_95.sum())} "
      f"({detach_95.mean():.1%})")
print(f"states separating from it at 99.8%           : {int(detach_998.sum())} "
      f"({detach_998.mean():.1%})")
print()
top = sorted(intervals, key=lambda r: r.point)[:3]
bottom = sorted(intervals, key=lambda r: -r.point)[:3]
print("the ends of the table, with their attainable spans:")
for r in top + bottom:
    print(f"  rank {r.point:>2}  {r.label:<22} could be {r.best}-{r.worst} (span {r.width})")


precision-weighted national ratio            : 0.8113
states separating from it at 95%             : 42 (82.4%)
states separating from it at 99.8%           : 29 (56.9%)

the ends of the table, with their attainable spans:
  rank  1  Vermont                could be 1-5 (span 5)
  rank  2  New York               could be 1-6 (span 6)
  rank  3  District of Columbia   could be 1-17 (span 17)
  rank 51  Wyoming                could be 49-51 (span 3)
  rank 50  Utah                   could be 49-51 (span 3)
  rank 49  Louisiana              could be 48-51 (span 4)


**42 of 51 states (82.4%)** separate from the precision-weighted national ratio of
**0.8113** at 95%, and **29 (56.9%)** at 99.8%. Detection against a benchmark works where
ordering does not, which is the pattern the whole series reports.

And the ends of the table resolve. Vermont at rank 1 could be anywhere in 1–5; Wyoming and Utah
at the bottom are pinned to 49–51. Between them sits the District of Columbia, third by point
estimate and able to occupy **any of ranks 1 through 17**.

That contrast is the notebook's practical summary. *Vermont is among the highest and Wyoming
among the lowest* is supported. *The District of Columbia is third* is not.

## Limitations

1. **A ratio of medians is not a median of ratios**, and it is not any individual's experience.
   It compares two summary statistics of two different distributions and says nothing about the
   shape of either.

2. **None of this is a pay-discrimination estimate.** No occupation, industry, hours beyond the
   full-time year-round flag, experience, education or tenure is adjusted for. The notebook is
   about what the published number is, not about what causes it.

3. **Section 5 establishes composition, not mechanism.** It shows women are less often in the
   full-time year-round group. It says nothing about why, and the reasons — which include
   constraint and choice in unmeasured proportions — are exactly what the table cannot see.

4. **The ratio MOE is a first-order approximation.** The Census formula is a delta-method
   linearisation. It is the published guidance and is used here for that reason, but it is not
   exact, and its accuracy degrades as the denominator's relative error grows.

5. **Medians carry an extra approximation.** ACS medians are interpolated from grouped
   distributions and their published margins reflect that interpolation. Two medians combined by
   a formula derived for ordinary estimates is standard practice and remains an approximation on
   top of an approximation.

6. **The `is_determined` bound is conservative by construction.** `rank_intervals` uses exact
   interval overlap with no distributional or independence assumption, so it cannot understate
   ambiguity. A simulation-based bound would be narrower and would rest on assumptions a
   published ACS table does not licence.

7. **States only, one year.** Nothing here establishes the result at county or tract level — the
   question §7.1 leaves open — and nothing here is a time series.

8. **Delaware's 1.004 is one state in one definition.** It is reported because it is in the data,
   not because one state above parity in the non-full-time group means anything on its own.

## Key Takeaways

1. **Three published ratios, one question.** 80.5, 72.4 and 87.7 cents, all from `B20017`, all
   defensible, all called the gender pay gap.

2. **The denominator is worth 8.1 cents** and moves the same way in 51 of 51 states. It also
   reorders states, from +1.3 cents in DC to +12.7 in Idaho.

3. **Check that a decomposition applies to your statistic.** A pooled ratio below both components
   proves differential composition for means and proves nothing for medians.

4. **Go and find the table with the counts.** `B20005` settles in one cell what `B20017` cannot
   settle at all.

5. **Conditioning on full-time year-round work conditions on a variable that differs by sex by
   11.3 points** — 71.1% of men against 60.1% of women, women 1.38× as likely to be excluded.

6. **λ = 0.9745, interval 10.78% of spread, 51 units — and 0 determined ranks.** The fourth case
   in §7.3, and the first with a relative spread of 0.2383, which is what that section asked for.

7. **79.6% of pairs separate and no rank is fixed.** A rank must win fifty comparisons, not one;
   0.796⁵⁰ ≈ 10⁻⁵. Report both numbers or the zero looks like imprecision.

8. **The extremes hold.** Vermont 1–5, Wyoming and Utah 49–51, DC anywhere in 1–17.

## Exercises

1. **Re-rank under the other definition.** Recompute section 7 on the all-with-earnings ratio.
   Report how many states change rank by more than five positions, and whether the number of
   determined ranks is still zero.

2. **Test the exponent claim.** Section 8 argues the zero follows from the unit count. Draw
   random subsets of 10, 20 and 40 states, recompute determined ranks in each, and plot the count
   against subset size. Does it behave like the heuristic predicts?

3. **Push to counties.** Run the same construction on `B20017` at county level. Predict, before
   running it, whether the pairwise separation rate or the determined-rank count moves more, then
   state which of §7.1's open questions your answer bears on.

4. **Decompose properly.** Section 4 shows medians do not decompose. Using `B20005`'s earnings
   brackets, construct a bounded estimate of what the all-earners ratio would be if women had
   men's full-time year-round share, and state exactly what that counterfactual does and does not
   mean.

5. **Check the approximation.** Section 5's limitation 4 notes the ratio MOE is a linearisation.
   Bootstrap the ratio interval for the three states with the largest denominator relative error
   and compare against the delta-method figure. Report the disagreement.

6. **Write the caveat that travels.** The headline is quoted far more often than its definition.
   In two sentences a newsroom would actually print, state the 80.5-cent figure and what it
   conditions on.

## References

- U.S. Census Bureau. *American Community Survey, table B20017: Median Earnings in the Past 12
  Months by Sex by Work Experience for the Population 16 Years and Over with Earnings.*
  https://data.census.gov/table/ACSDT5Y2023.B20017
- U.S. Census Bureau. *American Community Survey, table B20005: Sex by Work Experience by
  Earnings.* https://data.census.gov/table/ACSDT5Y2023.B20005
- U.S. Census Bureau. *Understanding and Using American Community Survey Data: What All Data
  Users Need to Know* — the ratio and proportion margin-of-error formulas used throughout.
  https://www.census.gov/programs-surveys/acs/library/handbooks/general.html
- Blau, F. D. and Kahn, L. M. (2017). "The Gender Wage Gap: Extent, Trends, and Explanations."
  *Journal of Economic Literature* 55(3): 789–865 — on what the unadjusted ratio does and does
  not measure.
- Goldin, C. (2014). "A Grand Gender Convergence: Its Last Chapter." *American Economic Review*
  104(4): 1091–1119 — on hours and temporal flexibility, which is the mechanism behind section 5's
  composition finding and is not tested here.
- Handbook §1.1 (margins of error), §3 (denominators and universes), §7.1 and §7.3 (precision and
  rank).
- `src/krl_tutorials_data/uncertainty.py` (`ratio_moe`) and `src/krl_tutorials_data/reliability.py`
  (`rank_intervals`).